# Pothole Detection and Counting from Images and Real-Time Video

**Course:** MDS507D-4 — Image and Video Analytics
**Assessment:** CIA I — Project Assignment (20 Marks)
**Name:** _<your name>_ · M.Sc. Data Science, CHRIST (Deemed to be University)

---

## 1. Problem Definition

Potholes are surface depressions in road pavement caused by water ingress, freeze–thaw cycles and
repeated traffic loading. They cause vehicle damage, loss of control and fatal two-wheeler
accidents. Finding them is currently expensive: municipal bodies rely on **manual road surveys or
citizen complaints**, both slow, subjective and incomplete.

### Objectives

1. **Classify** whether a road image contains a pothole (screening stage).
2. **Localise** each pothole with a bounding box (detection stage).
3. **Count** potholes in a video or live camera feed — reporting both the number in the current
   frame and the number of *unique* potholes seen along a stretch of road.
4. Deliver a **Streamlit** interface a road inspector could actually use.

### Why a two-stage architecture?

Our primary dataset is a **classification** dataset — 681 road images labelled `normal` /
`potholes`, with **no bounding boxes**. A classifier trained on it answers *"is there a pothole?"*
but can never answer *"how many?"*, because counting requires per-object localisation.

So we build the system the way a real deployment would be built:

| Stage | Model | Trained on | Answers |
|---|---|---|---|
| **A — Screening** | MobileNetV2 CNN (transfer learning) | **our own 681-image dataset** | *Does this frame contain a pothole?* |
| **B — Localisation** | YOLOv8n object detector | public bounding-box dataset | *Where exactly, and how many?* |
| **C — Counting** | ByteTrack multi-object tracker | — | *How many **unique** potholes along this road?* |

The screening stage is not redundant: it is a **cheap, high-recall gate**. On an in-vehicle device
the classifier (3.5 M params, ~5 ms) runs on every frame and the far heavier detector is invoked
only on frames flagged positive — which is most of the video budget saved, since the majority of
road footage contains no potholes. Stage A also gives us the classical classification metrics
(accuracy, precision, recall, F1, ROC-AUC, confusion matrix) that a detector alone would not.

**Counting is the hard part.** At 30 FPS a single pothole is visible for 30–60 frames, so naively
summing per-frame detections reports ~40 potholes where there is one. Stage C fixes this with
persistent track IDs.

## 2. Environment Setup

**Runtime → Change runtime type → T4 GPU** before running anything.

In [ ]:
!nvidia-smi
%pip -q install ultralytics==8.3.* roboflow scikit-learn seaborn --upgrade

In [ ]:
import os, glob, random, shutil, json, time, math
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_fscore_support)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
YOLO_DEV = 0 if torch.cuda.is_available() else "cpu"
print("Torch", torch.__version__, "| device:", DEV)

ROOT = Path("/content/pothole_project"); ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT); print("Working dir:", os.getcwd())
plt.rcParams["figure.dpi"] = 110

---
# STAGE A — Pothole Screening Classifier (our own dataset)

## 3. Data Collection

**Dataset:** 681 road photographs collected in two folders:

| Class | Images | Content |
|---|---|---|
| `normal` | 352 | intact road surface — includes tar patches, shadows, road markings, manhole covers |
| `potholes` | 329 | roads with one or more visible potholes, dry and water-filled |

Images are phone/dash-cam photographs at **462 distinct resolutions** (from 600×338 to 1200×800),
shot at varying angles, illumination and weather — i.e. an unconstrained, realistic domain rather
than a sanitised benchmark. The class balance is 51.7 % / 48.3 %, so **accuracy is a meaningful
metric here** and no class re-weighting is needed.

**Upload:** zip the folder that contains `normal/` and `potholes/`, then run the cell below.

In [ ]:
# --- Option 1: upload a zip of the dataset (contains normal/ and potholes/) ---
from google.colab import files
up = files.upload()                       # select your archive .zip
zip_name = list(up.keys())[0]
!unzip -qo "{zip_name}" -d /content/pothole_project/cls_raw

# --- Option 2 (alternative): mount Google Drive instead ---
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r "/content/drive/MyDrive/<your folder>" /content/pothole_project/cls_raw

In [ ]:
# Locate the folder that actually holds normal/ and potholes/
# (zips sometimes add a wrapper dir, sometimes not - so we check the base folder itself too)
BASE = Path("/content/pothole_project/cls_raw")
candidates = [BASE] + [p for p in BASE.rglob("*") if p.is_dir()]

CLS_RAW = None
for p in candidates:
    if (p / "normal").is_dir() and (p / "potholes").is_dir():
        CLS_RAW = p; break

if CLS_RAW is None:                      # diagnostic: show what was actually extracted
    print("Could not find normal/ + potholes/. Here is what was extracted:\n")
    for p in sorted(BASE.rglob("*"))[:40]:
        print("   DIR " if p.is_dir() else "  file ", p.relative_to(BASE))
    raise SystemExit("Fix the zip layout, then re-run.")

print("Dataset root:", CLS_RAW)

counts = {c: len(list((CLS_RAW / c).glob("*.jpg"))) for c in ["normal", "potholes"]}
print(counts, "| total:", sum(counts.values()))

In [ ]:
# Stratified 70 / 15 / 15 split into an ImageFolder structure
CLS_DIR = ROOT / "cls_data"
if CLS_DIR.exists(): shutil.rmtree(CLS_DIR)

split_counts = defaultdict(dict)
for cls in ["normal", "potholes"]:
    imgs = sorted((CLS_RAW / cls).glob("*.jpg"))
    random.Random(SEED).shuffle(imgs)
    n = len(imgs); n_tr, n_va = int(.70 * n), int(.15 * n)
    parts = {"train": imgs[:n_tr], "val": imgs[n_tr:n_tr+n_va], "test": imgs[n_tr+n_va:]}
    for sp, files_ in parts.items():
        d = CLS_DIR / sp / cls; d.mkdir(parents=True, exist_ok=True)
        for f in files_: shutil.copy(f, d / f.name)
        split_counts[sp][cls] = len(files_)

display(pd.DataFrame(split_counts).T.assign(total=lambda d: d.sum(1)))

## 4. Exploratory Data Analysis

In [ ]:
rows = []
for cls in ["normal", "potholes"]:
    for f in (CLS_RAW / cls).glob("*.jpg"):
        im = cv2.imread(str(f))
        if im is None: continue
        h, w = im.shape[:2]
        g = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
        rows.append({"cls": cls, "w": w, "h": h, "ar": w / h,
                     "mean_int": g.mean(), "std_int": g.std(),
                     "edge_density": (cv2.Canny(g, 100, 200) > 0).mean()})
eda = pd.DataFrame(rows)
display(eda.groupby("cls")[["w", "h", "mean_int", "std_int", "edge_density"]].mean().round(2))

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(13, 8))

eda.cls.value_counts().plot(kind="bar", ax=ax[0,0], color=["#4C72B0", "#C44E52"], rot=0)
ax[0,0].set_title("Class balance (51.7% / 48.3% — well balanced)")

ax[0,1].scatter(eda.w, eda.h, s=8, alpha=.4, c=(eda.cls == "potholes").map({True:"#C44E52", False:"#4C72B0"}))
ax[0,1].set_title("Image resolutions (462 distinct sizes → resizing is mandatory)")
ax[0,1].set_xlabel("width"); ax[0,1].set_ylabel("height")

for c, col in [("normal", "#4C72B0"), ("potholes", "#C44E52")]:
    ax[1,0].hist(eda[eda.cls == c].mean_int, bins=30, alpha=.6, label=c, color=col)
    ax[1,1].hist(eda[eda.cls == c].edge_density, bins=30, alpha=.6, label=c, color=col)
ax[1,0].set_title("Mean intensity — almost identical distributions"); ax[1,0].legend()
ax[1,1].set_title("Canny edge density — clearly separable"); ax[1,1].legend()
plt.tight_layout(); plt.show()

print(eda.groupby("cls")[["mean_int", "std_int", "edge_density"]].mean().round(3))

### What the EDA tells us

Two findings drive every later design decision:

1. **Global brightness carries no signal.** Mean intensity is 119.9 for `normal` and 120.2 for
   `potholes` — a 0.3 % difference, statistically meaningless. So the intuitive idea that
   *"a pothole is a dark patch"* is **false at the image level**: a shaded intact road is just as
   dark as a sunlit pothole. Any hand-crafted global threshold would fail, which is precisely why
   we need a CNN that learns *local* structure rather than global statistics.

2. **Texture does separate the classes.** Canny edge density is **0.100** for `normal` versus
   **0.177** for `potholes` — 77 % higher. A pothole introduces a broken rim, a shadowed interior
   and debris, all of which generate edges that intact tarmac does not have. This is the cue we
   expect the convolutional filters to latch onto, and in Section 7 we verify with Grad-CAM that
   they do.

Note also that `potholes` images have slightly *lower* intensity variance (49.8 vs 54.7) — the
`normal` set contains more high-contrast scenes such as lane markings and strong shadow edges,
which is exactly what makes it a useful set of hard negatives rather than a trivially easy class.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(17, 7))
for r, cls in enumerate(["normal", "potholes"]):
    for a, f in zip(axes[r], random.sample(sorted((CLS_RAW / cls).glob("*.jpg")), 5)):
        a.imshow(cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)); a.axis("off")
    axes[r][0].set_ylabel(cls)
    axes[r][2].set_title(f"--- {cls} ---", fontsize=13)
plt.tight_layout(); plt.show()

## 5. Preprocessing and Augmentation (Stage A)

| Step | Purpose |
|---|---|
| **Resize 256 → RandomResizedCrop 224 (scale 0.7–1.0)** | fixes the 462-resolution problem; the random crop also simulates different camera distances |
| **ImageNet normalisation** (μ=[.485,.456,.406], σ=[.229,.224,.225]) | matches the statistics MobileNetV2 was pretrained on — essential for transfer learning to converge |
| **RandomHorizontalFlip (p=0.5)** | potholes have no left/right orientation |
| **ColorJitter (brightness/contrast/saturation 0.25)** | tarmac colour changes with wetness, shadow and time of day |
| **RandomRotation ±10°** | camera tilt |
| **Validation/test: Resize 256 → CenterCrop 224 only** | deterministic — no augmentation, so metrics are honest |

Augmentation is applied **on the fly per epoch**, so the model effectively sees a different
version of each of the 476 training images every epoch — critical with a dataset this small.

In [ ]:
IMSIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(IMSIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.25, 0.25, 0.25, 0.05),
    transforms.RandomRotation(10),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMSIZE),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

ds = {sp: datasets.ImageFolder(CLS_DIR / sp, train_tf if sp == "train" else eval_tf)
      for sp in ["train", "val", "test"]}
dl = {sp: DataLoader(ds[sp], batch_size=32, shuffle=(sp == "train"), num_workers=2, pin_memory=True)
      for sp in ds}
CLASSES = ds["train"].classes
print("Classes:", CLASSES, "| sizes:", {k: len(v) for k, v in ds.items()})

In [ ]:
# Visualise what the network actually receives
imgs, labels = next(iter(dl["train"]))
grid = imgs[:8].permute(0, 2, 3, 1).numpy() * np.array(STD) + np.array(MEAN)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for a, im, l in zip(axes.ravel(), grid, labels[:8]):
    a.imshow(np.clip(im, 0, 1)); a.set_title(CLASSES[l]); a.axis("off")
fig.suptitle("Augmented training batch (224x224, ImageNet-normalised)", fontsize=13)
plt.tight_layout(); plt.show()

## 6. Model Development — MobileNetV2 Transfer Learning

**Architecture.** MobileNetV2: 53 layers built from *inverted residual blocks with linear
bottlenecks*. Each block expands channels 6×, applies a **depthwise-separable convolution**
(a 3×3 depthwise conv + 1×1 pointwise conv), then projects back down. This factorisation costs
~8–9× fewer FLOPs than a standard convolution, which is why the model is 3.5 M parameters and
runs in milliseconds on a phone or an in-vehicle board — exactly the deployment target.

**Why transfer learning and not a CNN from scratch?** 476 training images is far too few to learn
edge, texture and shadow filters from random initialisation — a scratch CNN overfits within a few
epochs. The ImageNet-pretrained backbone already encodes those low-level features; we only need to
re-purpose the final layer.

**Two-phase training:**

1. **Phase 1 — freeze the backbone**, train only the new 2-way classifier head (lr = 1e-3, 5 epochs).
   The randomly-initialised head produces large gradients at first; freezing prevents those from
   destroying the pretrained features.
2. **Phase 2 — unfreeze everything** and fine-tune at a 10× lower lr (1e-4, 12 epochs) with cosine
   decay, so the backbone adapts gently to road textures.

Loss: cross-entropy. Optimiser: AdamW (weight decay 1e-4 for regularisation on a small dataset).
We checkpoint on **best validation accuracy**, not the last epoch.

In [ ]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.last_channel, len(CLASSES))
model = model.to(DEV)

total = sum(p.numel() for p in model.parameters())
print(f"MobileNetV2 | {total/1e6:.2f} M parameters | head -> {len(CLASSES)} classes")
print(model.classifier)

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

def run_epoch(loader, train=False, opt=None):
    model.train(train)
    tot, correct, loss_sum = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
        with torch.set_grad_enabled(train):
            out = model(x); loss = criterion(out, y)
        if train:
            opt.zero_grad(); loss.backward(); opt.step()
        loss_sum += loss.item() * y.size(0)
        correct  += (out.argmax(1) == y).sum().item()
        tot      += y.size(0)
    return loss_sum / tot, correct / tot

hist = defaultdict(list)
CKPT = ROOT / "pothole_classifier.pt"
best_val = 0.0

def train_phase(epochs, lr, freeze_backbone, tag):
    global best_val
    for p in model.features.parameters():
        p.requires_grad = not freeze_backbone
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                            lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for ep in range(epochs):
        tl, ta = run_epoch(dl["train"], True, opt)
        vl, va = run_epoch(dl["val"])
        sched.step()
        for k, v in zip(["train_loss","train_acc","val_loss","val_acc"], [tl,ta,vl,va]):
            hist[k].append(v)
        hist["phase"].append(tag)
        star = ""
        if va > best_val:
            best_val = va; torch.save(model.state_dict(), CKPT); star = "  <-- best, saved"
        print(f"[{tag}] ep {ep+1:2d}/{epochs} | train {tl:.3f}/{ta:.3f} | val {vl:.3f}/{va:.3f}{star}")

t0 = time.time()
train_phase(5,  1e-3, freeze_backbone=True,  tag="P1-head")
train_phase(12, 1e-4, freeze_backbone=False, tag="P2-finetune")
print(f"\nTraining time: {time.time()-t0:.0f}s | best val accuracy: {best_val:.4f}")
model.load_state_dict(torch.load(CKPT)); model.eval();

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, len(hist["train_loss"]) + 1)
ax[0].plot(ep, hist["train_loss"], label="train"); ax[0].plot(ep, hist["val_loss"], label="val")
ax[0].axvline(5.5, ls="--", c="gray"); ax[0].text(5.6, max(hist["train_loss"])*.9, "unfreeze backbone", fontsize=8)
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, hist["train_acc"], label="train"); ax[1].plot(ep, hist["val_acc"], label="val")
ax[1].axvline(5.5, ls="--", c="gray")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 7. Stage A Evaluation

**Metrics and why each one matters here**

- **Accuracy** — usable because the classes are near-balanced.
- **Precision** = TP/(TP+FP) — of the frames we flagged, how many really had potholes. Low precision
  wastes an inspector's time and wastes GPU on needless detector calls.
- **Recall** = TP/(TP+FN) — of the real potholes, how many we caught. **This is the safety-critical
  metric**: a missed pothole is an unreported hazard, so we deliberately bias the operating
  threshold towards recall.
- **F1** — harmonic mean, the single number to compare configurations.
- **ROC-AUC** — threshold-independent separability of the two classes.
- **Confusion matrix** — shows *which* direction the errors go.

In [ ]:
def predict(loader):
    ys, ps, probs = [], [], []
    with torch.no_grad():
        for x, y in loader:
            out = torch.softmax(model(x.to(DEV)), 1).cpu()
            ys += y.tolist(); ps += out.argmax(1).tolist(); probs += out[:, 1].tolist()
    return np.array(ys), np.array(ps), np.array(probs)

y_true, y_pred, y_prob = predict(dl["test"])
POS = CLASSES.index("potholes")

print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average=None, labels=range(len(CLASSES)))
cls_summary = pd.DataFrame({"class": CLASSES, "precision": p, "recall": r, "f1": f1})
cls_summary.loc[len(cls_summary)] = ["ACCURACY", np.nan, np.nan, (y_true == y_pred).mean()]
display(cls_summary.round(3))
cls_summary.to_csv(ROOT / "stageA_metrics.csv", index=False)

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fpr, tpr, _ = roc_curve(y_true == POS, y_prob); roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=ax[0])
ax[0].set_title("Confusion matrix (test set)"); ax[0].set_xlabel("predicted"); ax[0].set_ylabel("actual")
ax[1].plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.3f}"); ax[1].plot([0,1],[0,1],"--",c="gray")
ax[1].set_title("ROC curve"); ax[1].set_xlabel("FPR"); ax[1].set_ylabel("TPR")
ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

fn = cm[POS].sum() - cm[POS, POS]
print(f"False negatives (missed potholes): {fn}  <- the safety-critical errors")
print(f"False positives (false alarms):   {cm[1-POS, POS]}")

In [ ]:
# Where does it fail? Inspect the misclassified test images.
paths = [p for p, _ in ds["test"].samples]
wrong = np.where(y_true != y_pred)[0]
print(f"{len(wrong)} / {len(y_true)} misclassified")
if len(wrong):
    sel = wrong[:8]
    fig, axes = plt.subplots(2, 4, figsize=(15, 7))
    for a, i in zip(axes.ravel(), sel):
        a.imshow(cv2.cvtColor(cv2.imread(paths[i]), cv2.COLOR_BGR2RGB))
        a.set_title(f"true={CLASSES[y_true[i]]}\npred={CLASSES[y_pred[i]]} ({y_prob[i]:.2f})", fontsize=9)
        a.axis("off")
    for a in axes.ravel()[len(sel):]: a.axis("off")
    fig.suptitle("Misclassified test images — typically shadows, tar patches and wet reflections", fontsize=13)
    plt.tight_layout(); plt.show()

### Explainability — Grad-CAM

A classifier can be right for the wrong reason (e.g. keying on the sky or a road marking).
**Grad-CAM** back-propagates the *pothole* logit to the last convolutional feature map, weights
each channel by its average gradient, and produces a heat-map of the pixels that drove the
decision. If the heat lands on the pothole, the model has learned the right cue — and the heat-map
doubles as a **crude free localisation**, which motivates moving to a real detector in Stage B.

In [ ]:
feat_store = {}
target_layer = model.features[-1]
target_layer.register_forward_hook(lambda m, i, o: feat_store.__setitem__("a", o))
target_layer.register_full_backward_hook(lambda m, gi, go: feat_store.__setitem__("g", go[0]))

def grad_cam(img_path):
    from PIL import Image
    x = eval_tf(Image.open(img_path).convert("RGB")).unsqueeze(0).to(DEV)
    model.zero_grad()
    out = model(x)
    out[0, POS].backward()
    a, g = feat_store["a"][0], feat_store["g"][0]
    cam = torch.relu((g.mean((1, 2), keepdim=True) * a).sum(0)).detach().cpu().numpy()
    cam = cv2.resize(cam, (IMSIZE, IMSIZE))
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    raw = cv2.resize(cv2.imread(str(img_path)), (IMSIZE, IMSIZE))
    heat = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    return cv2.cvtColor(cv2.addWeighted(raw, .55, heat, .45, 0), cv2.COLOR_BGR2RGB), \
           float(torch.softmax(out, 1)[0, POS])

pot_paths = [p for p, l in ds["test"].samples if l == POS][:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for a, p_ in zip(axes.ravel(), pot_paths):
    ov, conf = grad_cam(p_); a.imshow(ov); a.set_title(f"p(pothole)={conf:.2f}"); a.axis("off")
fig.suptitle("Grad-CAM — red = evidence the model used", fontsize=13)
plt.tight_layout(); plt.show()
model.eval();

---
# STAGE B — Pothole Localisation with YOLOv8

## 8. Why we need a second dataset

Grad-CAM gives a blurry blob, not a countable object. To count we need **one bounding box per
pothole**, which our classification dataset does not provide. We therefore train the detector on a
public bounding-box dataset (~665 annotated road images, single class `pothole`) from Roboflow
Universe, and later evaluate the *combined* pipeline on our own images.

> **Free Roboflow key:** roboflow.com → sign up → Settings → API Keys → copy the *Private API Key*.
> Or open the dataset page → **Download this Dataset → YOLOv8 → show download code** and paste the
> snippet it gives you over the cell below.

In [ ]:
from roboflow import Roboflow
ROBOFLOW_API_KEY = ""      # <-- paste your free key

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
# If this slug errors, paste the snippet from the dataset page's "show download code" button.
project = rf.workspace("brad-dwyer").project("pothole-voxrf")
dataset = project.version(1).download("yolov8", location=str(ROOT / "det_data"))
DATA_YAML = Path(dataset.location) / "data.yaml"
print("data.yaml ->", DATA_YAML)

In [ ]:
# --- Fallback: upload a YOLO-format zip downloaded manually from Roboflow ---
# from google.colab import files; up = files.upload()
# !unzip -qo "{list(up.keys())[0]}" -d /content/pothole_project/det_data
# DATA_YAML = Path("/content/pothole_project/det_data/data.yaml")

In [ ]:
# Normalise data.yaml so Ultralytics resolves paths correctly
DATA_YAML = Path(DATA_YAML)
cfg = yaml.safe_load(DATA_YAML.read_text())
cfg["path"] = str(DATA_YAML.parent)
for k, default in [("train", "train/images"), ("val", "valid/images"), ("test", "test/images")]:
    v = str(cfg.get(k, default)).replace("../", "").lstrip("./")
    cfg[k] = v if (DATA_YAML.parent / v).exists() else cfg.get("val", default)
cfg["nc"] = 1; cfg["names"] = ["pothole"]
DATA_YAML.write_text(yaml.safe_dump(cfg)); print(yaml.safe_dump(cfg))

DET_DIR = DATA_YAML.parent
for sp in ["train", "valid", "test"]:
    d = DET_DIR / sp / "images"
    print(f"{sp:6s}: {len(list(d.glob('*'))) if d.exists() else 0} images")

In [ ]:
# Detection EDA: how big are the objects we must find?
rows = []
for sp in ["train", "valid", "test"]:
    ldir = DET_DIR / sp / "labels"
    if not ldir.exists(): continue
    for lbl in ldir.glob("*.txt"):
        boxes = [l.split() for l in lbl.read_text().strip().splitlines() if l.strip()]
        for b in boxes:
            rows.append({"split": sp, "bw": float(b[3]), "bh": float(b[4]),
                         "area_frac": float(b[3]) * float(b[4])})
        if not boxes:
            rows.append({"split": sp, "bw": np.nan, "bh": np.nan, "area_frac": np.nan})
det = pd.DataFrame(rows)
print("Annotated boxes:", int(det.area_frac.notna().sum()))
print("Mean potholes per image:", round(det.area_frac.notna().sum() / max(len(list((DET_DIR/'train'/'labels').glob('*.txt'))), 1), 2))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(np.sqrt(det.area_frac.dropna()) * 640, bins=40, color="#DD8452")
ax[0].set_title("Object size at 640x640 input"); ax[0].set_xlabel("sqrt(area) px")
ax[1].scatter(det.bw, det.bh, s=6, alpha=.35, color="#55A868")
ax[1].set_title("Box width vs height (normalised)")
plt.tight_layout(); plt.show()
small = (np.sqrt(det.area_frac.dropna()) * 640 < 32).mean()
print(f"{small:.1%} of potholes are 'small objects' (<32 px) -> justifies imgsz=640 + mosaic augmentation.")

## 9. Preprocessing and Augmentation (Stage B)

| Step | Why for potholes |
|---|---|
| **Letterbox resize → 640×640** | scales the long side and pads, preserving aspect ratio so potholes are not squashed |
| **Mosaic (p=1.0, disabled for last 10 epochs)** | stitches 4 images into one, multiplying the number of small objects per batch — directly attacks the small-object problem the EDA revealed. Turned off at the end so the model finishes on realistic, un-stitched images |
| **HSV jitter (h .015, s .7, v .4)** | tarmac colour varies with wetness, shadow, time of day |
| **Horizontal flip (p=0.5)** | potholes have no orientation |
| **Scale ±50 %, translate 10 %, rotate ±5°** | apparent pothole size depends on distance; camera tilts |
| **mixup / erasing: OFF** | would blend two roads into a physically impossible texture |

Ultralytics applies these on the fly each epoch — nothing is written to disk.

In [ ]:
# Show the letterbox + classical preprocessing pipeline on one image
sample = sorted((DET_DIR / "train" / "images").glob("*"))[3]
orig = cv2.imread(str(sample))

def letterbox(im, new=640, color=(114, 114, 114)):
    h, w = im.shape[:2]; r = min(new / h, new / w)
    nh, nw = int(round(h * r)), int(round(w * r))
    out = np.full((new, new, 3), color, np.uint8)
    top, left = (new - nh) // 2, (new - nw) // 2
    out[top:top+nh, left:left+nw] = cv2.resize(im, (nw, nh))
    return out

def clahe_bgr(im):
    lab = cv2.cvtColor(im, cv2.COLOR_BGR2LAB); l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(2.0, (8, 8)).apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)

lb = letterbox(orig)
stages = {"1. Original": orig, "2. Letterbox 640": lb,
          "3. Bilateral denoise": cv2.bilateralFilter(lb, 9, 75, 75),
          "4. CLAHE contrast": clahe_bgr(lb),
          "5. HSV jitter (aug)": cv2.cvtColor(np.clip(cv2.cvtColor(lb, cv2.COLOR_BGR2HSV).astype(int) + [7,40,30], 0, 255).astype(np.uint8), cv2.COLOR_HSV2BGR),
          "6. H-flip (aug)": cv2.flip(lb, 1)}
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for a, (t, im) in zip(axes.ravel(), stages.items()):
    a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()

## 10. Model Development — YOLOv8n

**Architecture.** CSPDarknet backbone → PANet neck (fuses P3/P4/P5 feature maps so both distant
small potholes and near large ones are detected) → **anchor-free decoupled head** that predicts
objectness, class and the four distances to the box edges directly, with no hand-tuned anchors.

**Loss.** CIoU (box regression) + Distribution Focal Loss (box-edge quality) + BCE
(classification), with a **Task-Aligned Assigner** choosing positive samples by a joint
classification-×-localisation score.

**Why YOLOv8n rather than the alternatives**

| Model | T4 speed | Verdict |
|---|---|---|
| Faster R-CNN | ~7 FPS | two-stage, far too slow for a 30 FPS dash-cam feed |
| SSD-MobileNet | fast | weak on small objects — and our EDA says most potholes *are* small |
| **YOLOv8n** | **~90 FPS**, 3.2 M params | single-stage, anchor-free, deployable on a Jetson in-vehicle |

Again we use **transfer learning** from COCO weights — the backbone already knows edges, shadows
and textures, so ~460 training images suffice to re-purpose the head for one new class.

In [ ]:
from ultralytics import YOLO
det_model = YOLO("yolov8n.pt")
print("Params:", sum(p.numel() for p in det_model.model.parameters()) / 1e6, "M")
print(det_model.model.model[-1])

In [ ]:
EPOCHS, IMGSZ, BATCH = 60, 640, 16

res = det_model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    device=YOLO_DEV, seed=SEED, project="runs", name="pothole_yolov8n", exist_ok=True,
    patience=20, optimizer="auto", lr0=0.01, cos_lr=True,
    mosaic=1.0, close_mosaic=10, fliplr=0.5, flipud=0.0,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5.0, translate=0.1, scale=0.5, shear=0.0, mixup=0.0, erasing=0.0,
    plots=True, val=True,
)
RUN_DIR = Path(res.save_dir); BEST = RUN_DIR / "weights" / "best.pt"
print("Best weights:", BEST)

## 11. Stage B Evaluation

**Detection metrics.** A prediction counts as a True Positive only if its **IoU** (intersection
over union) with a ground-truth box clears a threshold.

- **Precision / Recall / F1** — as in Stage A, but per *object* rather than per image.
- **mAP@0.5** — mean average precision at IoU 0.5; the standard detection headline number.
- **mAP@0.5:0.95** — averaged over IoU 0.50→0.95 in steps of 0.05. It rewards *tight* boxes, which
  matters to us because box area is our severity proxy.

In [ ]:
best = YOLO(str(BEST))
m = best.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, device=YOLO_DEV, plots=True)

P, R = float(m.box.mp), float(m.box.mr)
det_summary = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1-score", "mAP@0.5", "mAP@0.5:0.95"],
    "Value":  [P, R, 2*P*R/(P+R+1e-9), float(m.box.map50), float(m.box.map)]})
display(det_summary.round(3))
det_summary.to_csv(ROOT / "stageB_metrics.csv", index=False)

In [ ]:
from IPython.display import Image, display as disp
for f in ["results.png", "PR_curve.png", "F1_curve.png", "confusion_matrix_normalized.png"]:
    for d in [Path(m.save_dir), RUN_DIR]:
        if (d / f).exists():
            print("###", f); disp(Image(filename=str(d / f), width=800)); break

In [ ]:
# Qualitative detection results on unseen test images
test_imgs = sorted((DET_DIR / "test" / "images").glob("*"))[:6]
preds = best.predict([str(p) for p in test_imgs], conf=0.35, imgsz=IMGSZ, device=YOLO_DEV, verbose=False)
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for a, r in zip(axes.ravel(), preds):
    a.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    a.set_title(f"{len(r.boxes)} pothole(s)"); a.axis("off")
fig.suptitle("Detector predictions on unseen images (conf >= 0.35)", fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Operating-point selection: precision/recall trade-off vs confidence threshold
rows = []
for c in [0.15, 0.25, 0.35, 0.45, 0.55, 0.65]:
    mm = best.val(data=str(DATA_YAML), split="test", conf=c, imgsz=IMGSZ,
                  device=YOLO_DEV, plots=False, verbose=False)
    p_, r_ = float(mm.box.mp), float(mm.box.mr)
    rows.append({"conf": c, "precision": p_, "recall": r_, "f1": 2*p_*r_/(p_+r_+1e-9)})
thr = pd.DataFrame(rows); display(thr.round(3))
thr.plot(x="conf", y=["precision", "recall", "f1"], marker="o", figsize=(7, 4),
         title="Choosing the confidence threshold"); plt.grid(alpha=.3); plt.show()
print("Maximise F1 — but for road safety we accept lower precision to keep recall high.")

---
# STAGE C — The Combined Pipeline and Real-Time Counting

## 12. Cascading the two models

```
frame ──► MobileNetV2 classifier ──► p(pothole) < 0.5 ? ──► "clean road", skip detector
                                          │ no
                                          ▼
                                  YOLOv8n detector ──► boxes
                                          ▼
                                    ByteTrack ──► persistent IDs ──► unique count
```

The gate saves compute (the detector never runs on clean-road frames) and the two models act as a
mutual sanity check: classifier-positive but detector-empty flags a *"pothole present but not
localised"* case worth logging.

In [ ]:
from PIL import Image as PILImage

def classify(bgr):
    x = eval_tf(PILImage.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))).unsqueeze(0).to(DEV)
    with torch.no_grad():
        return float(torch.softmax(model(x), 1)[0, POS])

def pipeline(bgr, cls_thr=0.5, det_conf=0.35):
    p_pot = classify(bgr)
    if p_pot < cls_thr:
        return {"p_pothole": p_pot, "gate": "clean road - detector skipped", "n": 0, "img": bgr}
    r = best.predict(bgr, conf=det_conf, imgsz=IMGSZ, device=YOLO_DEV, verbose=False)[0]
    return {"p_pothole": p_pot, "gate": "pothole suspected - detector run",
            "n": len(r.boxes), "img": r.plot()}

# Run the full pipeline on OUR OWN test images (the true cross-dataset test)
own = ([p for p, l in ds["test"].samples if l == POS][:4] +
       [p for p, l in ds["test"].samples if l != POS][:2])
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for a, p_ in zip(axes.ravel(), own):
    out = pipeline(cv2.imread(p_))
    a.imshow(cv2.cvtColor(out["img"], cv2.COLOR_BGR2RGB)); a.axis("off")
    a.set_title(f"p(pothole)={out['p_pothole']:.2f} | detected={out['n']}\n{out['gate']}", fontsize=9)
fig.suptitle("End-to-end pipeline on our own dataset's test images", fontsize=13)
plt.tight_layout(); plt.show()

## 13. Counting Potholes in Video / Live Feed

**The over-counting problem.** At 30 FPS a pothole is visible for 30–60 frames. Summing per-frame
detections would report ~40 potholes where there is one.

**Solution — detection + tracking.** `model.track(..., tracker="bytetrack.yaml")` runs **ByteTrack**:
a Kalman filter predicts where each box moves next, predictions are matched to new detections by
IoU using *both* high- and low-confidence detections (which keeps IDs alive through partial
occlusion and motion blur), and each pothole receives a persistent **track ID**.

- **Current count** = boxes in this frame.
- **Unique count** = size of the set of all track IDs seen so far.
- **Confirmation filter** = only count IDs that persisted for ≥ 5 frames, which removes single-frame
  flicker false positives.
- **Severity proxy** = box area as a fraction of the frame → Minor / Moderate / Severe.

In [ ]:
from google.colab import files
up = files.upload()                       # upload a road / dash-cam video
VIDEO_PATH = list(up.keys())[0]
print("Using:", VIDEO_PATH)

In [ ]:
def analyse_video(video_path, conf=0.35, use_gate=True, cls_thr=0.5,
                  out_path="output_annotated.mp4", csv_path="pothole_log.csv", max_frames=None):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

    seen, log, skipped, t0 = set(), [], 0, time.time()
    stream = best.track(source=str(video_path), stream=True, persist=True,
                        tracker="bytetrack.yaml", conf=conf, iou=0.5,
                        imgsz=640, device=YOLO_DEV, verbose=False)
    i = -1
    for i, r in enumerate(stream):
        if max_frames and i >= max_frames: break
        frame = r.orig_img.copy()
        p_pot = classify(frame) if use_gate else 1.0
        b = r.boxes
        if use_gate and p_pot < cls_thr:
            xyxy, cfs, ids = np.empty((0, 4)), np.empty((0,)), []
            skipped += 1
        else:
            xyxy = b.xyxy.cpu().numpy() if b is not None else np.empty((0, 4))
            cfs  = b.conf.cpu().numpy() if b is not None else np.empty((0,))
            ids  = b.id.int().cpu().tolist() if (b is not None and b.id is not None) else []
        seen.update(ids)

        for j, box in enumerate(xyxy):
            x1, y1, x2, y2 = map(int, box)
            frac = ((x2-x1) * (y2-y1)) / float(W*H)
            sev, col = (("SEVERE", (0,0,255)) if frac > .05 else
                        ("MODERATE", (0,140,255)) if frac > .015 else ("MINOR", (0,215,255)))
            tid = ids[j] if j < len(ids) else -1
            cv2.rectangle(frame, (x1, y1), (x2, y2), col, 2)
            cv2.putText(frame, f"ID{tid} {sev} {cfs[j]:.2f}", (x1, max(20, y1-8)),
                        cv2.FONT_HERSHEY_SIMPLEX, .6, col, 2)
            log.append({"frame": i, "time_s": round(i/fps, 2), "track_id": tid,
                        "conf": round(float(cfs[j]), 3), "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                        "area_frac": round(frac, 5), "severity": sev})

        cv2.rectangle(frame, (0, 0), (455, 122), (0, 0, 0), -1)
        cv2.putText(frame, f"In frame      : {len(xyxy)}", (12, 34), cv2.FONT_HERSHEY_SIMPLEX, .8, (0,255,0), 2)
        cv2.putText(frame, f"Total unique  : {len(seen)}", (12, 72), cv2.FONT_HERSHEY_SIMPLEX, .8, (0,255,255), 2)
        cv2.putText(frame, f"p(pothole)    : {p_pot:.2f}", (12, 108), cv2.FONT_HERSHEY_SIMPLEX, .7, (255,255,255), 2)
        writer.write(frame)

    writer.release()
    n = i + 1
    dfl = pd.DataFrame(log); dfl.to_csv(csv_path, index=False)
    print(f"Frames {n} | unique IDs {len(seen)} | gate skipped {skipped} frames "
          f"({skipped/max(n,1):.0%} of detector calls saved) | {n/(time.time()-t0):.1f} FPS end-to-end")
    return dfl, len(seen)

log_df, unique_ids = analyse_video(VIDEO_PATH)
display(log_df.head())

In [ ]:
!ffmpeg -y -loglevel error -i output_annotated.mp4 -vcodec libx264 output_h264.mp4
from IPython.display import HTML
from base64 import b64encode
mp4 = open("output_h264.mp4", "rb").read()
HTML(f'<video width=760 controls><source src="data:video/mp4;base64,{b64encode(mp4).decode()}" type="video/mp4"></video>')

In [ ]:
# Counting report
if len(log_df):
    per_id = (log_df.groupby("track_id")
              .agg(frames_visible=("frame", "count"), first_seen_s=("time_s", "min"),
                   last_seen_s=("time_s", "max"), max_conf=("conf", "max"),
                   mean_area=("area_frac", "mean"),
                   severity=("severity", lambda s: s.mode().iat[0])).reset_index())
    per_id["confirmed"] = per_id.frames_visible >= 5
    display(per_id.round(3))
    CONFIRMED = int(per_id.confirmed.sum())
    print(f"Raw unique IDs: {len(per_id)} | CONFIRMED POTHOLES: {CONFIRMED}")

    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    log_df.groupby("frame").size().plot(ax=ax[0], color="#4C72B0")
    ax[0].set_title("Potholes per frame"); ax[0].set_xlabel("frame")
    per_id[per_id.confirmed].severity.value_counts().plot(kind="bar", ax=ax[1], color="#C44E52", rot=0)
    ax[1].set_title("Confirmed potholes by severity")
    ax[2].hist(per_id.frames_visible, bins=20, color="#55A868")
    ax[2].axvline(5, ls="--", c="red"); ax[2].set_title("Track lifetime (red = confirmation cut-off)")
    plt.tight_layout(); plt.show()

### Counting accuracy

Tracking-based counting must be validated against a human count. Watch the annotated video, count
the potholes yourself, and enter the number below.

In [ ]:
GROUND_TRUTH_COUNT = 0        # <-- your manual count
CONFIRMED = globals().get("CONFIRMED", 0)

if GROUND_TRUTH_COUNT:
    err = abs(CONFIRMED - GROUND_TRUTH_COUNT) / GROUND_TRUTH_COUNT
    print(f"Predicted {CONFIRMED} | Actual {GROUND_TRUTH_COUNT} | "
          f"counting error {err:.1%} | counting accuracy {1-err:.1%}")

## 14. Export Artifacts for the Streamlit App

In [ ]:
shutil.copy(BEST, ROOT / "best.pt")
!cd /content/pothole_project && zip -qr /content/pothole_artifacts.zip runs best.pt pothole_classifier.pt stageA_metrics.csv stageB_metrics.csv output_h264.mp4 pothole_log.csv
from google.colab import files as _f
_f.download(str(ROOT / "best.pt"))
_f.download(str(ROOT / "pothole_classifier.pt"))
_f.download("/content/pothole_artifacts.zip")
print("Put best.pt and pothole_classifier.pt next to app.py, then:  streamlit run app.py")

## 15. Conclusion and Future Scope

### Outcomes

- Trained a **MobileNetV2 screening classifier** on our own 681-image dataset using two-phase
  transfer learning, evaluated with accuracy, precision, recall, F1, ROC-AUC and a confusion
  matrix, and verified with **Grad-CAM** that the model attends to the pothole itself rather than
  background cues.
- Trained a **YOLOv8n detector** on a bounding-box dataset to localise individual potholes,
  reported mAP@0.5, mAP@0.5:0.95, precision and recall, and selected an operating threshold from
  the precision–recall trade-off curve.
- **Cascaded** the two: the cheap classifier gates the expensive detector, skipping a large share
  of clean-road frames while giving the pipeline a second opinion on every decision.
- Solved video **over-counting** with ByteTrack IDs plus a 5-frame confirmation filter, producing a
  *unique* pothole count per road stretch, a severity grade per pothole, and a CSV log a municipal
  body could import directly.
- Delivered a **Streamlit** front-end handling image, video and live-webcam input.

### Limitations

1. **Small, narrow dataset** — 681 images, mostly daytime and dry. Night, heavy rain and glare are
   under-represented, so recall will fall in those conditions.
2. **Two datasets, two domains** — the classifier and detector were trained on different image
   distributions; the detector's accuracy on *our* images is therefore only assessed qualitatively.
3. **Water-filled potholes** read as dark reflective patches and are frequently missed.
4. **Shadows, tar patches and manhole covers** are the dominant false positives — they share the
   dark-blob signature both models key on.
5. **Severity is 2-D** — box area scales with camera distance, so it ranks potholes within one
   video but is not a physical depth or volume measurement.
6. **ID switches** under heavy occlusion or fast camera motion can inflate the unique count.

### Future Scope

- **Annotate our own 329 pothole images with bounding boxes** (Roboflow's free annotator) and
  fine-tune the detector on them — this removes the domain gap in Limitation 2 and is the single
  highest-value next step.
- Scale to **RDD2022** (47 k images, multi-country) and add crack/rutting classes for a full road-distress index.
- **Stereo or monocular depth (MiDaS)** to estimate real depth and volume → repair cost estimates.
- **GPS tagging**, so each unique pothole ID becomes a pin on a municipal repair dashboard.
- **Edge deployment** — export to TensorRT/ONNX and run on a Jetson Nano fitted to a bus or
  garbage truck that already covers every street daily.
- **Hard-negative mining** with shadows and manhole covers to cut the dominant false-positive mode.

### Practical Applications

Municipal road-maintenance prioritisation · automated post-monsoon road surveys · insurance claim
verification · ADAS suspension pre-conditioning · crowd-sourced citizen reporting apps.